# 🚁 Drone Tello EDU - Exploration de Bâtiments Délabrés

## Système d'exploration autonome avec cartographie thermique

Ce notebook démontre les capacités du système d'exploration optimisé pour :
- **Navigation autonome sécurisée** avec scans 360° périodiques
- **Cartographie spatiale et thermique** simultanée
- **Évitement d'obstacles** fixes et mobiles avec réaction rapide
- **Détection de zones dangereuses** (feu, trous, débris)

---

## 1. Installation et Configuration

In [ ]:
# Installation des dépendances
!pip install -q djitellopy numpy opencv-python matplotlib

In [ ]:
# Imports
import sys
import os
import time
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from IPython.display import display, clear_output, HTML
import logging

# Configuration du logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Imports du projet
from tello_controller import TelloController, DroneState
from exploration import ExplorationMission, MissionConfig, MissionStatus
from vision import VideoStream, ObstacleDetector, ThermalDetector
from mapping import DualMap, ExplorationPlanner
from obstacle_avoidance import ObstacleAvoidanceSystem, ThreatLevel

print("✅ Modules chargés avec succès!")
print(f"📁 Répertoire: {os.getcwd()}")

## 2. Test des Composants Individuels

### 2.1 Contrôleur du Drone

In [ ]:
# Test du contrôleur en mode simulation
print("=" * 50)
print("TEST DU CONTRÔLEUR")
print("=" * 50)

controller = TelloController(simulation_mode=True)
controller.connect()

print(f"\n📊 État initial: {controller.state.value}")
print(f"📍 Position: {controller.position.to_tuple()}")

# Décollage
controller.takeoff()
print(f"\n🛫 Après décollage: {controller.position.to_tuple()}")

# Séquence de mouvements
movements = [
    ("Avance 100cm", lambda: controller.move_forward(100)),
    ("Rotation 90°", lambda: controller.rotate_clockwise(90)),
    ("Avance 50cm", lambda: controller.move_forward(50)),
    ("Monte 30cm", lambda: controller.move_up(30)),
]

print("\n📐 Séquence de mouvements:")
for name, action in movements:
    action()
    print(f"  {name}: Position = {controller.position.to_tuple()}, Yaw = {controller.position.yaw:.0f}°")

# Télémétrie
telemetry = controller.get_telemetry()
print(f"\n📡 Télémétrie:")
for k, v in telemetry.items():
    print(f"  {k}: {v}")

controller.land()
controller.disconnect()
print("\n✅ Test contrôleur terminé")

### 2.2 Vision et Détection d'Obstacles

In [ ]:
import cv2

print("=" * 50)
print("TEST DE LA VISION")
print("=" * 50)

# Initialisation
video = VideoStream(simulation_mode=True)
detector = ObstacleDetector()
thermal = ThermalDetector()

video.start()
time.sleep(1)

# Capture et analyse
frame = video.get_frame()

if frame is not None:
    # Détection
    obstacles = detector.detect(frame)
    thermal_map, hotspots = thermal.detect(frame)
    
    # Résultats
    print(f"\n📷 Frame capturée: {frame.shape}")
    print(f"🚧 Obstacles détectés: {len(obstacles)}")
    print(f"🌡️ Points chauds: {len(hotspots)}")
    print(f"🔥 Température max: {thermal.get_max_temperature():.1f}°C")
    
    # Affichage détails obstacles
    if obstacles:
        print("\n   Détails obstacles:")
        for i, obs in enumerate(obstacles[:5]):
            print(f"   [{i+1}] {obs.obstacle_type.value}: {obs.distance_estimate:.0f}cm - Menace: {obs.threat_level.value}")
    
    # Visualisation
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Image originale
    axes[0].imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    axes[0].set_title("Vue Caméra (Simulée)")
    axes[0].axis('off')
    
    # Détections
    frame_detections = detector.draw_detections(frame, obstacles)
    axes[1].imshow(cv2.cvtColor(frame_detections, cv2.COLOR_BGR2RGB))
    axes[1].set_title(f"Détection Obstacles ({len(obstacles)})")
    axes[1].axis('off')
    
    # Carte thermique
    if thermal_map is not None:
        axes[2].imshow(cv2.cvtColor(thermal_map, cv2.COLOR_BGR2RGB))
        axes[2].set_title(f"Vue Thermique (Max: {thermal.get_max_temperature():.0f}°C)")
        axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()

video.stop()
print("\n✅ Test vision terminé")

### 2.3 Système d'Évitement d'Obstacles

In [ ]:
print("=" * 50)
print("TEST ÉVITEMENT D'OBSTACLES")
print("=" * 50)

avoidance = ObstacleAvoidanceSystem()
avoidance.start_monitoring()

# Ajout d'obstacles
print("\n🚧 Ajout d'obstacles de test:")

obs1 = avoidance.add_obstacle(100, 50, 100, 112, (0.89, 0.45, 0), "debris")
print(f"  - Débris à (100, 50, 100)")

obs2 = avoidance.add_obstacle(-50, 80, 100, 94, (-0.53, 0.85, 0), "person")
print(f"  - Personne à (-50, 80, 100)")

# Simuler mouvement obstacle mobile
obs3 = avoidance.add_obstacle(0, 150, 100, 150, (0, 1, 0), "unknown")
for i in range(5):
    time.sleep(0.1)
    obs3.update_position(obs3.x + 15, obs3.y + 5, obs3.z, obs3.distance)

print(f"  - Obstacle mobile: vitesse = {obs3.get_velocity()}")

# Test collision
drone_pos = (0, 0, 100)
target_pos = (120, 60, 100)

print(f"\n🎯 Test trajectoire: {drone_pos} → {target_pos}")

risk, obstacle, threat = avoidance.check_collision_risk(*drone_pos, *target_pos)

print(f"  Risque collision: {risk}")
print(f"  Niveau menace: {threat.value} ({threat.name})")

if risk and obstacle:
    strategy = avoidance.get_avoidance_strategy(drone_pos, target_pos, obstacle)
    print(f"  Stratégie: {strategy.value}")
    
    path = avoidance.calculate_avoidance_path(drone_pos, target_pos, obstacle, strategy)
    print(f"  Chemin évitement: {[f'({p[0]:.0f}, {p[1]:.0f}, {p[2]:.0f})' for p in path]}")

# Visualisation
fig, ax = plt.subplots(figsize=(10, 8))

# Obstacles
for obs in avoidance.detected_obstacles:
    color = 'red' if obs.is_mobile else 'orange'
    circle = plt.Circle((obs.x, obs.y), 30, color=color, alpha=0.5)
    ax.add_patch(circle)
    ax.annotate(obs.obstacle_type, (obs.x, obs.y + 40), ha='center', fontsize=8)

# Trajectoire
ax.plot([drone_pos[0], target_pos[0]], [drone_pos[1], target_pos[1]], 
        'b--', linewidth=2, label='Trajectoire directe')

if risk and 'path' in dir():
    path_x = [drone_pos[0]] + [p[0] for p in path]
    path_y = [drone_pos[1]] + [p[1] for p in path]
    ax.plot(path_x, path_y, 'g-', linewidth=2, label="Chemin d'évitement")

# Drone et cible
ax.plot(*drone_pos[:2], 'b^', markersize=15, label='Drone')
ax.plot(*target_pos[:2], 'g*', markersize=15, label='Cible')

ax.set_xlim(-200, 200)
ax.set_ylim(-50, 250)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.legend(loc='upper right')
ax.set_title("Planification d'évitement d'obstacles")
ax.set_xlabel("X (cm)")
ax.set_ylabel("Y (cm)")

plt.show()

# Stats
print(f"\n📊 Statistiques:")
stats = avoidance.get_status_report()
print(f"  Total obstacles: {stats['total_obstacles']}")
print(f"  Obstacles mobiles: {stats['mobile_obstacles']}")
print(f"  Temps réaction moyen: {stats['stats']['avg_reaction_time_ms']:.2f}ms")

avoidance.stop_monitoring()
print("\n✅ Test évitement terminé")

## 3. Mission d'Exploration Complète

### 3.1 Configuration de la Mission

In [ ]:
# Configuration de la mission
config = MissionConfig(
    # Zone d'exploration (3m x 3m)
    area_width=300,
    area_height=300,
    
    # Altitude de vol
    exploration_altitude=100,  # 1 mètre
    
    # Précision
    step_size=50,  # 50cm entre points
    
    # Pattern d'exploration
    pattern="snake",  # Options: snake, spiral, room_search
    
    # Sécurité
    scan_interval=200,    # Scan 360° tous les 2m
    safety_margin=80,     # Marge de sécurité 80cm
    min_battery=15,       # Arrêt si batterie < 15%
    
    # Fonctionnalités
    enable_mapping=True,
    enable_thermal=True,
    enable_avoidance=True,
    enable_scanning=True,
    
    # Durée max
    max_duration=120  # 2 minutes
)

print("📋 Configuration de la mission:")
print(f"  Zone: {config.area_width/100:.1f}m x {config.area_height/100:.1f}m")
print(f"  Altitude: {config.exploration_altitude}cm")
print(f"  Pattern: {config.pattern}")
print(f"  Scan tous les: {config.scan_interval}cm")
print(f"  Marge sécurité: {config.safety_margin}cm")

### 3.2 Initialisation et Préparation

In [ ]:
# Création de la mission en mode simulation
mission = ExplorationMission(config, simulation_mode=True)

# Variables de suivi
mission_events = []
waypoints_reached = []
thermal_alerts = []

# Callbacks
def on_status_change(old, new):
    event = f"[{time.strftime('%H:%M:%S')}] Status: {old.value} → {new.value}"
    mission_events.append(event)
    print(event)

def on_waypoint(wp, progress):
    waypoints_reached.append(wp)
    if len(waypoints_reached) % 5 == 0:  # Afficher tous les 5 waypoints
        print(f"  ✓ Waypoint ({wp[0]:.0f}, {wp[1]:.0f}) - Progression: {progress:.1f}%")

def on_thermal(pos, temp, hotspots):
    thermal_alerts.append((pos, temp, time.time()))
    print(f"  🔥 ALERTE THERMIQUE: {temp:.0f}°C à ({pos.x:.0f}, {pos.y:.0f})")

def on_obstacle(obs):
    print(f"  ⚠️ Obstacle: {obs.obstacle_type} à ({obs.x:.0f}, {obs.y:.0f}) - {obs.distance:.0f}cm")

def on_scan_complete(results):
    print("  📡 Scan 360° terminé:")
    for direction in ['front', 'right', 'back', 'left']:
        data = results[direction]
        status = "✓" if data['clear'] else "⚠️"
        print(f"      {direction}: {status} {data['distance']:.0f}cm")

# Configuration callbacks
mission.on_status_change = on_status_change
mission.on_waypoint_reached = on_waypoint
mission.on_thermal_alert = on_thermal
mission.on_obstacle_detected = on_obstacle
mission.on_scan_complete = on_scan_complete

# Préparation
print("\n" + "=" * 60)
print("PRÉPARATION DE LA MISSION")
print("=" * 60)

if mission.prepare_mission():
    print(f"\n✅ Mission prête!")
    print(f"   Waypoints planifiés: {mission.total_waypoints}")
else:
    print("❌ Échec de la préparation")

### 3.3 Configuration de l'Environnement (Obstacles et Zones Thermiques)

In [ ]:
print("\n" + "=" * 60)
print("CONFIGURATION ENVIRONNEMENT SIMULÉ")
print("=" * 60)

# Obstacles fixes (débris)
print("\n🚧 Ajout d'obstacles:")
mission.add_simulated_obstacle(80, 50, 100, obstacle_type="debris")
print("  - Débris à (80, 50)")

mission.add_simulated_obstacle(-60, 80, 100, obstacle_type="debris")
print("  - Débris à (-60, 80)")

# Obstacle mobile (personne)
mission.add_simulated_obstacle(0, 120, 100, is_mobile=True, velocity=(8, 3, 0), obstacle_type="person")
print("  - Personne mobile à (0, 120) vitesse=(8, 3)")

# Trou au sol
mission.dual_map.add_obstacle(-30, -50, 0, radius=40, obstacle_type="hole", threat_level=4)
print("  - Trou au sol à (-30, -50)")

# Zones thermiques
print("\n🌡️ Ajout de zones thermiques:")

# Foyer d'incendie
mission.dual_map.add_thermal_zone(100, 80, 50, radius=60, temperature=180, is_active=True)
print("  - Feu actif à (100, 80) - 180°C")

# Zone chaude (braises)
mission.dual_map.add_thermal_zone(-80, 60, 30, radius=40, temperature=85, is_active=False)
print("  - Braises à (-80, 60) - 85°C")

# Zone tiède
mission.dual_map.add_thermal_zone(50, -70, 20, radius=50, temperature=45, is_active=False)
print("  - Zone tiède à (50, -70) - 45°C")

print("\n✅ Environnement configuré")

### 3.4 Exécution de la Mission

In [ ]:
print("\n" + "=" * 60)
print("🚀 DÉMARRAGE DE L'EXPLORATION")
print("=" * 60)

mission.start_exploration()

# Suivi en temps réel
start_time = time.time()
duration = 15  # Durée de la simulation (secondes)

try:
    while mission.status == MissionStatus.IN_PROGRESS and (time.time() - start_time) < duration:
        time.sleep(1)
        
        # Affichage périodique
        if int(time.time() - start_time) % 5 == 0:
            pos = mission.controller.position
            progress = mission.planner.get_progress()
            print(f"\n  📍 T+{int(time.time() - start_time)}s: ({pos.x:.0f}, {pos.y:.0f}, {pos.z:.0f}) | {progress:.1f}%")

except KeyboardInterrupt:
    print("\n⚠️ Interruption utilisateur")

print("\n" + "=" * 60)
print("ARRÊT DE LA MISSION")
print("=" * 60)

mission.stop_exploration()

print(f"\n📊 Résumé:")
print(f"  Durée: {time.time() - start_time:.1f}s")
print(f"  Waypoints atteints: {len(waypoints_reached)}")
print(f"  Alertes thermiques: {len(thermal_alerts)}")

## 4. Visualisation des Résultats

### 4.1 Cartes ASCII

In [ ]:
print("\n" + "=" * 60)
print("CARTE D'ALTITUDE")
print("=" * 60)
mission.display_map(show_thermal=False)

print("\n" + "=" * 60)
print("CARTE THERMIQUE")
print("=" * 60)
mission.display_map(show_thermal=True)

### 4.2 Visualisation Graphique

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# 1. Carte d'altitude
ax1 = axes[0, 0]
altitude_data = np.nan_to_num(mission.dual_map.altitude_grid, nan=0)
im1 = ax1.imshow(altitude_data, cmap='terrain', origin='lower', 
                  extent=[-mission.config.area_width/2, mission.config.area_width/2,
                          -mission.config.area_height/2, mission.config.area_height/2])
plt.colorbar(im1, ax=ax1, label='Altitude (cm)')
ax1.set_title('Carte d\'Altitude')
ax1.set_xlabel('X (cm)')
ax1.set_ylabel('Y (cm)')

# Ajouter obstacles
for obs in mission.dual_map.obstacles:
    color = 'red' if obs.is_mobile else 'black'
    ax1.plot(obs.x, obs.y, 'X', color=color, markersize=10)

# 2. Carte thermique
ax2 = axes[0, 1]
im2 = ax2.imshow(mission.dual_map.thermal_grid, cmap='hot', origin='lower',
                  extent=[-mission.config.area_width/2, mission.config.area_width/2,
                          -mission.config.area_height/2, mission.config.area_height/2],
                  vmin=20, vmax=200)
plt.colorbar(im2, ax=ax2, label='Température (°C)')
ax2.set_title('Carte Thermique')
ax2.set_xlabel('X (cm)')
ax2.set_ylabel('Y (cm)')

# Marquer zones thermiques
for zone in mission.dual_map.thermal_zones:
    circle = plt.Circle((zone.x, zone.y), zone.radius, 
                         fill=False, color='yellow', linewidth=2)
    ax2.add_patch(circle)

# 3. Grille d'occupation
ax3 = axes[1, 0]
occupancy = mission.dual_map.occupancy_grid.copy()
cmap_occ = plt.cm.colors.ListedColormap(['lightgray', 'lightgreen', 'red'])
bounds = [-1.5, -0.5, 0.5, 100.5]
norm = plt.cm.colors.BoundaryNorm(bounds, cmap_occ.N)
im3 = ax3.imshow(occupancy, cmap=cmap_occ, norm=norm, origin='lower',
                  extent=[-mission.config.area_width/2, mission.config.area_width/2,
                          -mission.config.area_height/2, mission.config.area_height/2])
ax3.set_title('Grille d\'Occupation')
ax3.set_xlabel('X (cm)')
ax3.set_ylabel('Y (cm)')

# Légende
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='lightgray', label='Inexploré'),
    Patch(facecolor='lightgreen', label='Libre'),
    Patch(facecolor='red', label='Occupé')
]
ax3.legend(handles=legend_elements, loc='upper right')

# 4. Trajectoire
ax4 = axes[1, 1]
if waypoints_reached:
    traj_x = [wp[0] for wp in waypoints_reached]
    traj_y = [wp[1] for wp in waypoints_reached]
    
    # Ligne de trajectoire
    ax4.plot(traj_x, traj_y, 'b-', linewidth=1, alpha=0.7, label='Trajectoire')
    
    # Points avec couleur selon progression
    colors = np.linspace(0, 1, len(traj_x))
    ax4.scatter(traj_x, traj_y, c=colors, cmap='viridis', s=20)
    
    # Départ et fin
    ax4.plot(traj_x[0], traj_y[0], 'go', markersize=12, label='Départ')
    ax4.plot(traj_x[-1], traj_y[-1], 'r^', markersize=12, label='Fin')

# Obstacles sur trajectoire
for obs in mission.dual_map.obstacles:
    circle = plt.Circle((obs.x, obs.y), obs.radius, 
                         color='red' if obs.is_mobile else 'orange', 
                         alpha=0.5)
    ax4.add_patch(circle)

ax4.set_xlim(-mission.config.area_width/2 - 50, mission.config.area_width/2 + 50)
ax4.set_ylim(-mission.config.area_height/2 - 50, mission.config.area_height/2 + 50)
ax4.set_aspect('equal')
ax4.grid(True, alpha=0.3)
ax4.legend(loc='upper right')
ax4.set_title('Trajectoire d\'Exploration')
ax4.set_xlabel('X (cm)')
ax4.set_ylabel('Y (cm)')

plt.tight_layout()
plt.savefig('/tmp/exploration_results.png', dpi=150)
plt.show()

print("\n💾 Graphiques sauvegardés: /tmp/exploration_results.png")

### 4.3 Rapport de Mission

In [ ]:
print("\n" + "=" * 60)
print("📊 RAPPORT DE MISSION DÉTAILLÉ")
print("=" * 60)

report = mission.get_mission_report()

print(f"\n🎯 Statut: {report['status']}")
print(f"⏱️ Durée: {report['duration_seconds']:.1f}s")
print(f"🔋 Mode: {'Simulation' if report['simulation'] else 'Réel'}")

print(f"\n📍 Progression:")
print(f"   Waypoints: {report['waypoints']['completed']}/{report['waypoints']['total']}")
print(f"   Complétion: {report['waypoints']['progress']:.1f}%")

print(f"\n🗺️ Cartographie:")
print(f"   Couverture: {report['mapping']['coverage']:.1f}%")
print(f"   Points enregistrés: {report['mapping']['points_recorded']}")
print(f"   Obstacles détectés: {report['mapping']['obstacles_count']}")

print(f"\n🌡️ Thermique:")
print(f"   Température max: {report['thermal']['max_temperature']:.1f}°C")
print(f"   Feu détecté: {'⚠️ OUI' if report['thermal']['fire_detected'] else 'Non'}")
print(f"   Zones thermiques: {report['thermal']['zones']}")

print(f"\n🚧 Évitement:")
print(f"   Obstacles surveillés: {report['avoidance']['total_obstacles']}")
print(f"   Obstacles mobiles: {report['avoidance']['mobile_obstacles']}")
print(f"   Collisions évitées: {report['avoidance']['stats']['collisions_avoided']}")
print(f"   Temps réaction: {report['avoidance']['stats']['avg_reaction_time_ms']:.2f}ms")

print(f"\n🚁 Drone:")
print(f"   Position finale: {report['drone']['position']}")
print(f"   Orientation: {report['drone']['yaw']:.0f}°")
print(f"   État: {report['drone']['state']}")
print(f"   Batterie: {report['drone']['battery']}%")

## 5. Export des Données

In [ ]:
# Export des résultats
output_dir = "/tmp/exploration_data"
os.makedirs(output_dir, exist_ok=True)

mission.export_results(f"{output_dir}/mission")

print(f"\n💾 Données exportées dans {output_dir}:")
for f in os.listdir(output_dir):
    size = os.path.getsize(f"{output_dir}/{f}")
    print(f"   📄 {f} ({size/1024:.1f} KB)")

## 6. Analyse des Zones Dangereuses

In [ ]:
print("\n" + "=" * 60)
print("⚠️ ANALYSE DES ZONES DANGEREUSES")
print("=" * 60)

dangers = mission.dual_map.get_danger_zones()

if dangers:
    print(f"\n🚨 {len(dangers)} zones dangereuses identifiées:\n")
    
    for i, (x, y, danger_type) in enumerate(dangers, 1):
        print(f"   [{i}] {danger_type.upper()}")
        print(f"       Position: ({x:.0f}, {y:.0f}) cm")
        print(f"       Distance du départ: {(x**2 + y**2)**0.5:.0f} cm")
        
        # Recommandation
        if 'fire' in danger_type:
            print(f"       ⚡ Action: Éviter - Zone de combustion active")
        elif 'hole' in danger_type:
            print(f"       ⚡ Action: Éviter - Risque d'effondrement")
        elif 'hot' in danger_type:
            print(f"       ⚡ Action: Prudence - Température élevée")
        print()
else:
    print("\n✅ Aucune zone dangereuse critique identifiée")

# Visualisation des zones dangereuses
fig, ax = plt.subplots(figsize=(10, 8))

# Fond avec grille thermique
ax.imshow(mission.dual_map.thermal_grid, cmap='hot', origin='lower', alpha=0.5,
          extent=[-mission.config.area_width/2, mission.config.area_width/2,
                  -mission.config.area_height/2, mission.config.area_height/2])

# Zones dangereuses
for x, y, dtype in dangers:
    if 'fire' in dtype:
        color, marker, size = 'red', '*', 200
    elif 'hole' in dtype:
        color, marker, size = 'black', 'X', 150
    else:
        color, marker, size = 'orange', 'o', 100
    
    ax.scatter(x, y, c=color, marker=marker, s=size, label=dtype, zorder=5)

# Trajectoire sûre suggérée
ax.plot(0, 0, 'g^', markersize=15, label='Point de départ')

ax.set_xlim(-mission.config.area_width/2 - 30, mission.config.area_width/2 + 30)
ax.set_ylim(-mission.config.area_height/2 - 30, mission.config.area_height/2 + 30)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.set_title('Carte des Zones Dangereuses')
ax.set_xlabel('X (cm)')
ax.set_ylabel('Y (cm)')

# Légende sans doublons
handles, labels = ax.get_legend_handles_labels()
by_label = dict(zip(labels, handles))
ax.legend(by_label.values(), by_label.keys(), loc='upper right')

plt.tight_layout()
plt.show()

## 7. Conclusion

### Résumé des Capacités du Système

In [ ]:
print("\n" + "=" * 60)
print("📋 RÉSUMÉ DES CAPACITÉS DU SYSTÈME")
print("=" * 60)

capabilities = {
    "Navigation Autonome": [
        "✅ Décollage/Atterrissage automatique",
        "✅ Navigation par waypoints (snake, spiral, room_search)",
        "✅ Retour automatique à la base",
        "✅ Gestion de l'orientation (yaw tracking)"
    ],
    "Sécurité": [
        "✅ Scan 360° périodique (configurable)",
        "✅ Vérification continue des 6 axes",
        "✅ Réaction réflexe (<100ms)",
        "✅ Arrêt d'urgence instantané"
    ],
    "Évitement d'Obstacles": [
        "✅ Détection obstacles fixes",
        "✅ Tracking obstacles mobiles",
        "✅ Prédiction de trajectoire",
        "✅ Stratégies d'évitement multiples"
    ],
    "Cartographie": [
        "✅ Carte d'altitude en temps réel",
        "✅ Carte thermique simultanée",
        "✅ Grille d'occupation",
        "✅ Export JSON/NPY"
    ],
    "Détection Thermique": [
        "✅ Identification zones chaudes",
        "✅ Détection de feu actif",
        "✅ Alertes température",
        "✅ Visualisation thermique"
    ]
}

for category, features in capabilities.items():
    print(f"\n🔹 {category}:")
    for feature in features:
        print(f"   {feature}")

print("\n" + "=" * 60)
print("✅ DÉMONSTRATION TERMINÉE")
print("=" * 60)